# Naive RAG Experiments

Runs the baseline Naive RAG pipeline across different vector databases and embedding models.
This serves as the control experiment for comparison with Hybrid and Multi-Agent approaches.

**Pipeline:** Query → Dense Retrieval (Cosine/MMR) → LLM Generation → Answer  
**Variables:** Vector DB (Chroma, Qdrant, LanceDB), Embedding model (MiniLM, BGE)  
**Evaluation:** RAGAS and DeepEval metrics

In [5]:
import sys
sys.path.append("..")

import os
import time
import json
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
import config
from ast import literal_eval
from deepeval.evaluate import DisplayConfig, AsyncConfig
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric,
)
import deepeval
import instructor
from groq import AsyncGroq

from ragas.llms.base import InstructorLLM
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_13902/689212763.py:16: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/tmp/ipykernel_13902/689212763.py:16: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.

## LLM & RAG Chain

In [11]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty — set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        """Return a ChatGroq instance using the current API key."""
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        """Advance to the next key, wrapping around."""
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")


def is_rate_limit_error(e):
    msg = str(e).lower()
    return any(kw in msg for kw in [
        "rate_limit", "rate limit", "429", "too many requests",
        "tokens per", "token limit", "exceeded",
    ])


VANILLA_PROMPT_TEMPLATE = """Answer the following biomedical research question using your existing knowledge.

Question: {question}

Provide a precise, evidence-based answer. Do not speculate beyond established scientific knowledge. Give answer in 1 or 2 sentences

Answer:"""

VANILLA_PROMPT = PromptTemplate(
    template=VANILLA_PROMPT_TEMPLATE,
    input_variables=["question"],
)


def build_vanilla_llm_chain(llm):
    return VANILLA_PROMPT | llm


def _run_slice(slice_df, api_key, model, delay, key_idx):
    """Process a contiguous slice of the evaluation set using a single dedicated API key.

    Called in its own thread by run_rag_parallel. Rows are processed sequentially
    with `delay` seconds between requests to stay within the key's daily quota.
    Adds generated_answer as new column to a copy of slice_df.

    Args:
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        Copy of slice_df with one new columns: generated_answer.
    """
    chain = build_vanilla_llm_chain(ChatGroq(model=model, api_key=api_key))
    result_df = slice_df.copy().reset_index(drop=True)
    generated_answer_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        try:
            result = chain.invoke({"question": question})
            generated_answer_list[row_idx] = result.content
        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["generated_answer"] = generated_answer_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(slice_df)} rows collected")
    return result_df


def run_llm_parallel(df, key_rotator, rows_per_key=None, delay=None):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Key 0 gets rows 0..rows_per_key-1, key 1 gets the next rows_per_key rows, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    ThreadPoolExecutor is used (not asyncio) because: (1) network I/O releases the GIL
    so threads genuinely run concurrently, (2) Jupyter/Colab already have a running event
    loop so asyncio.run() raises RuntimeError, and (3) ChatGroq.invoke() is synchronous.

    Args:
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows within each key's slice (default config.PARALLEL_DELAY_SECONDS).

    Returns:
        a copy of df with one new column - generated_answer - in original row order.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(
            f"Warning: {len(df)} rows exceed capacity ({len(api_keys)} keys × {rows_per_key} rows = "
            f"{total_capacity}). Only the first {total_capacity} rows will be processed."
        )
        df = df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                s, key, key_rotator.model, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback = s.copy().reset_index(drop=True)
                fallback["generated_answer"] = [None] * len(s)
                ordered_results[idx] = fallback

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    return final_df

## Evaluation Functions

In [21]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    """Return a minimal DataFrame with question_index and metric score."""
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    """Evaluate with a non-LLM RAGAS metric over each row of the eval DataFrame.

    Uses SingleTurnSample + single_turn_score (synchronous) — no API keys required.
    Compares golden answer and generated answer.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
            Expected columns: question, generated_answer,
            golden_contexts, golden_answer.
        metric: A RAGAS metric instance.
        results_file: Optional CSV path to save per-sample scores (question_index + score only).

    Returns:
        Tuple of (list of per-sample scores, average score, scores DataFrame).
    """
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def build_ragas_combined(eval_df, score_dfs, results_file=None):
    """Combine eval_df with per-metric score DataFrames into one summary CSV.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
        score_dfs: List of score DataFrames from evaluate_ragas, each with
            question_index + one metric score column.
        results_file: Optional CSV path to save the combined DataFrame.

    Returns:
        Combined DataFrame with question_index, question,
        golden_contexts, golden_answer, generated_response, and one column per metric.
    """
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})

    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")

    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")

    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx):
    """Evaluate a contiguous slice of test cases using a single dedicated API key.

    Called in its own thread by evaluate_deepeval_parallel. Test cases are evaluated
    sequentially with `delay` seconds between each to stay within the key's daily quota.

    Args:
        test_case_slice: List of LLMTestCase objects assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        threshold: Pass/fail threshold for the metric.
        delay: Seconds to sleep between test cases.
        key_idx: Key index used only for log prefixes.

    Returns:
        List of deepeval test results in slice order.
    """
    llm = ChatGroq(model=model, api_key=api_key)
    results = []

    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm))
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config= DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}/{len(test_case_slice)}: {e}")

        if i < len(test_case_slice) - 1:
            time.sleep(delay)

    print(f"[Key {key_idx}] Done — {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None):
    """Assign a contiguous slice of test cases to each API key and run all slices in parallel.

    Key 0 gets test_cases[0:rows_per_key], key 1 gets the next slice, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    Args:
        test_cases: List of LLMTestCase objects built by build_test_cases.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        threshold: Pass/fail threshold for the metric (default 0.5).
        results_file: Optional CSV path to save per-sample scores.
        delay: Seconds between cases within each slice (defaults to config.DEEPEVAL_DELAY_SECONDS).
        rows_per_key: Max cases assigned to each key (default config.PARALLEL_BUCKET_SIZE).

    Returns:
        List of deepeval test results in original case order.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if not test_cases:
        raise ValueError("test_cases is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        print(
            f"Warning: {len(test_cases)} cases exceed capacity ({len(api_keys)} keys × {rows_per_key} = "
            f"{total_capacity}). Only the first {total_capacity} cases will be processed."
        )
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s) ({rows_per_key} cases/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: cases {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} cases)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_deepeval_slice,
                s, key, key_rotator.model,
                metric_cls, threshold, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")

        if results_file:
            rows = []
            for r in all_results:
                rows.append({
                    "question": r.input,
                    "generated_answer": r.actual_output,
                    "golden_answer": r.expected_output,
                    r.metrics_data[0].name: r.metrics_data[0].score,
                })
            pd.DataFrame(rows).to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results

---
## Setup

In [8]:
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Initialized GroqKeyRotator with 11 API key(s)
LLM: llama-3.3-70b-versatile
Run timestamp: 20260505_175240


## Prepare Evaluation Sample

Reads the pre-built golden dataset from `data/processed/golden_dataset_complete.csv`.
Generate this file once by running `python3 sampling.py` from the `research/` directory.
Keeping the split fixed is critical — regenerating mid-experiment would change which
questions each RAG variant sees, invalidating cross-experiment comparisons.

In [9]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
print(f"Structure of golden dataset")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
print(type(golden_df['golden_contexts'].iloc[0]))
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
Structure of golden dataset
<class 'list'>
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Run Vanilla LLM Evaluation

In [15]:
eval_dataset = run_llm_parallel(golden_df, key_rotator)
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"vanilla_llm_{timestamp}.csv"))
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 9] Done — 20/20 rows collected
[Key 5] Done — 20/20 rows collected
[Key 1] Done — 20/20 rows collected
[Key 2] Done — 20/20 rows collected
[Key 3] Done — 20/20 rows collected
[Key 7] Done — 20/20 rows collected
[Key 6] Done — 20/20 rows collected
[Key 0] Done — 20/20 rows collected
[Key 4] Done — 20/20 rows collected
[Key 8] Done — 20/20 rows collected

Completed 200/200 questions total
Generated 200 answers


---
## RAGAS Evaluation

In [16]:
ragas_blue_scores, ragas_blue_avg, ragas_blue_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"vanilla_llm_bleu_{timestamp}.csv")
)


=== BleuScore: 0.0757 (avg over 200 samples) ===
Saved scores to /content/results/ragas/vanilla_llm_bleu_20260505_175240.csv


In [17]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"vanilla_llm_rouge_{timestamp}.csv")
)


=== RougeScore: 0.2113 (avg over 200 samples) ===
Saved scores to /content/results/ragas/vanilla_llm_rouge_20260505_175240.csv


In [18]:
combined_ragas = build_ragas_combined(
    eval_dataset, [ragas_blue_df, ragas_rouge_df], results_file=str(config.RESULTS_RAGAS_DIR / f"vanilla_llm_combined_{timestamp}.csv"))

Saved combined RAGAS results to /content/results/ragas/vanilla_llm_combined_20260505_175240.csv


## DeepEval Evaluation

In [19]:
test_cases = build_test_cases(eval_dataset)

de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 11 API key(s)


In [23]:
deepeval_ar = evaluate_deepeval_parallel(
    test_cases, AnswerRelevancyMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"vanilla_llm_ans_relevancy_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


⚠ WARNING: No hyperparameters logged.
» ]8;id=237433;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.33s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

...

Warning: Could not update test run on disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

Warning: Could not load test run from disk: [Errno 2] No such file or directory: 
'.deepeval/.temp_test_run_data.json'

⚠ WARNING: No hyperparameters logged.
» ]8;id=653962;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.51s | token cost: None)
» Test Results (4 total tests):
   » Pass Rate: 100.0% | Passed: 4 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

[Key 8] Done — 20/20 cases evaluated

=== Answer Relevancy: 0.9806 (avg over 198 samples) ===
Saved to /content/results/deepeval/vanilla_llm_ans_relevancy_20260505_175240.csv
